> 在线讲义：[第 23 章 · 推理 kernel：decode 为什么是 memory-bound，以及 paged KV、split-KV、MLA、megakernel](https://wikieden.github.io/cuda-to-tensor-core/lessons/23-inference-kernels.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 23 章 · 推理 kernel：decode 为什么是 memory-bound，以及 paged KV、split-KV、MLA、megakernel

本章回答：大模型生成一个 token 时，GPU 的 Tensor Core 为什么大部分在闲着？KV cache 怎么像操作系统的分页内存一样管理？一条很长的序列怎么分给很多 SM 一起算 attention？DeepSeek 的 MLA 在 kernel 层面省了什么？megakernel 又是为了解决什么？

配套讲义：仓库里 `cuda-programming/notes/lessons/23-inference-kernels.md`（讲解、动画、练习都在讲义里）。对应原文：04 调研 §3.3（FlashInfer、DeepGEMM / FlashMLA）、§3.5（megakernel：HazyResearch「Look Ma, No Bubbles!」、MPK arXiv 2512.22219）；FlashAttention-2 §2.3（online softmax 合并，`materials/text/flashattention-2.txt` 第 138–200 行）；`distributed-pretraining/materials/code/DeepGEMM-README.md`；本仓库 llm-full-stack 推理部分；PagedAttention（vLLM 论文，〔旧知〕）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 decode 强度 | $2bKN / 2KN = b$ | `decode_intensity(1, 4096, 4096)` | 1 FLOP/byte |
| 第一步 速度上限 | 16 GB ÷ 3.35 TB/s | `ms_per_token(16e9)` | 4.8 ms，约 209 token/s（讲义"约 210"） |
| 第二步 KV cache | 2 × 32 × 8 × 128 × 2 B | `kv_bytes_per_token(32, 8, 128)` | 128 KB；8K token = 1 GB |
| 第二步 paged | 页表查物理位置 | `paged_gather(pool, [7, 2, 5], 40, 16)` | 取回的 K 和连续存放的一样 |
| 第三步 split-KV | 按 (m, ℓ) 合并 4 段 | `split_kv(q, K, V, 4)` vs `attention(q, K, V)` | 一致 |
| 第五步 INT4 | 字节 1/4 | `ms_per_token(4e9)` | 1.2 ms，上限提高 4 倍 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `attention`**（来自 `ch22_flashattention.py`）：标准写法：S = QKᵀ（N × N 整张落地）→ P = softmax(S) → O = PV。这里不除 √d，和讲义的公式一致。

In [ ]:
def attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    """标准写法：S = QKᵀ（N × N 整张落地）→ P = softmax(S) → O = PV。这里不除 √d，和讲义的公式一致。"""
    S = Q @ K.T
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P /= P.sum(axis=1, keepdims=True)
    return P @ V

**函数 `decode_intensity`**（来自 `ch23_inference_kernels.py`）：[b × K] × [K × N]：FLOP = 2bKN，字节 ≈ 权重 KN × 2（b 很小时激活可以忽略）。

In [ ]:
def decode_intensity(b: int, k: int, n: int, bytes_per: int = 2) -> float:
    """[b × K] × [K × N]：FLOP = 2bKN，字节 ≈ 权重 KN × 2（b 很小时激活可以忽略）。"""
    return 2 * b * k * n / (k * n * bytes_per)

**函数 `kv_bytes_per_token`**（来自 `ch23_inference_kernels.py`）：K 和 V 两份 × 层数 × KV 头数 × 头维度 × 每元素字节。

In [ ]:
def kv_bytes_per_token(layers: int, kv_heads: int, head_dim: int, bytes_per: int = 2) -> int:
    """K 和 V 两份 × 层数 × KV 头数 × 头维度 × 每元素字节。"""
    return 2 * layers * kv_heads * head_dim * bytes_per

**函数 `ms_per_token`**（来自 `ch23_inference_kernels.py`）：每个 token 至少把全部权重读一遍：权重字节 ÷ 带宽。

In [ ]:
def ms_per_token(weight_bytes: float, bw: float = 3.35e12) -> float:
    """每个 token 至少把全部权重读一遍：权重字节 ÷ 带宽。"""
    return weight_bytes / bw * 1e3

**函数 `paged_gather`**（来自 `ch23_inference_kernels.py`）：按页表把一条序列的 K（或 V）从显存池里取出来：token t 在第 t // page 页、页内第 t % page 个。

In [ ]:
def paged_gather(pool: np.ndarray, block_table: list[int], length: int, page: int) -> np.ndarray:
    """按页表把一条序列的 K（或 V）从显存池里取出来：token t 在第 t // page 页、页内第 t % page 个。"""
    t = np.arange(length)
    phys = np.array(block_table)[t // page] * page + t % page
    return pool[phys]

**函数 `split_kv`**（来自 `ch23_inference_kernels.py`）：把 K、V 沿长度切成几段，每段算局部输出 O_s 和 (m_s, ℓ_s)，再合并：O = Σ ℓ_s e^(m_s − m) O_s / ℓ。

In [ ]:
def split_kv(q: np.ndarray, K: np.ndarray, V: np.ndarray, splits: int) -> np.ndarray:
    """把 K、V 沿长度切成几段，每段算局部输出 O_s 和 (m_s, ℓ_s)，再合并：O = Σ ℓ_s e^(m_s − m) O_s / ℓ。"""
    parts = []
    for Ks, Vs in zip(np.array_split(K, splits), np.array_split(V, splits)):
        s = q @ Ks.T
        m = s.max(axis=-1)
        p = np.exp(s - m[..., None])
        parts.append((m, p.sum(axis=-1), p @ Vs / p.sum(axis=-1)[..., None]))   # 每段已经归一化的 O_s
    m = np.max([pm for pm, _, _ in parts], axis=0)
    w = [pl * np.exp(pm - m) for pm, pl, _ in parts]
    l = np.sum(w, axis=0)
    return sum(wi[..., None] * po for wi, (_, _, po) in zip(w, parts)) / l[..., None]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 decode：batch = 1 时的算术强度（K = N = 4096）")
print("  ≈", decode_intensity(1, 4096, 4096), "FLOP/byte，H100 脊点 295")
t = ms_per_token(16e9)
print(f"  Llama-3-8B（16 GB BF16）：每个 token 至少 {t:.1f} ms → batch 1 上限约 {1e3 / t:.0f} token/s")

In [ ]:
print("\n第二步 KV cache（32 层、8 个 KV 头、头维度 128、BF16）")
per = kv_bytes_per_token(32, 8, 128)
print(f"  每 token {per // 1024} KB；8K token 的序列 = {per * 8192 / 2**30:.0f} GB")

In [ ]:
print("\n第二步 paged attention：一条 40 个 token 的序列，每页 16 个 token，用了 3 页")
rng = np.random.default_rng(0)
d, page = 8, 16
K = rng.standard_normal((40, d))
pool = rng.standard_normal((10 * page, d))
table = [7, 2, 5]
for logical, phys in enumerate(table):
    n = min(page, 40 - logical * page)
    pool[phys * page:phys * page + n] = K[logical * page:logical * page + n]
print("  按页表取回来的 K 和连续存放的一样：", np.array_equal(paged_gather(pool, table, 40, page), K))
print("  token 20 在逻辑第", 20 // page, "页、页内第", 20 % page, "个 → 物理位置", table[20 // page] * page + 20 % page)

In [ ]:
print("\n第三步 split-KV：一个 query、2000 个 token，切 4 段")
q = rng.standard_normal((1, 64))
K2, V2 = rng.standard_normal((2000, 64)) / 8, rng.standard_normal((2000, 64))
print("  合并结果和一次算完一样：", np.allclose(split_kv(q, K2, V2, 4), attention(q, K2, V2)))

In [ ]:
print("\n第五步 INT4 权重：字节数降为 1/4")
print(f"  16 GB → 4 GB：每个 token 至少 {ms_per_token(4e9):.1f} ms，上限提高 {ms_per_token(16e9) / ms_per_token(4e9):.0f} 倍")

## 3. 代码里才看得到的

1. **`pool` 里其他页是别人的数据**：模拟里用随机数填满显存池，只有页表指到的 3 页是这条序列的；取错一页结果就对不上。
2. **split-KV 合并的是"已经归一化"的 O_s**：所以权重是 $\ell_s e^{m_s - m}$，最后再除以总的 ℓ；如果每段存的是没除 ℓ 的 O，合并式要相应改。
3. **K 除以 8 只是为了数值好看**：让分数不至于太大，和算法无关。

## 4. 自己改一改

- 把 `split_kv` 的段数换成 1、7、2000（每段一个 token），确认结果都一样。
- 把 `paged_gather` 的页大小换成 8，重新排页表，确认还能取回同样的 K。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> prefill 算不过来，decode 搬不过来：batch 为 1 时强度约为 1；
> 每个 token 至少读一遍全部权重：权重字节 ÷ 带宽 = 速度上限；
> KV cache 分页：页表查地址，不怕碎片；
> 长序列 split-KV：分段各算，按 (m, ℓ) 合并；
> MLA 缓存 latent、吸收投影，KV 小了强度高了；
> 量化要在 GEMM 里反量化；小 kernel 太多就上 Graph 或 megakernel。